# Task 2 – Domain-Shift Correction
**Owner:** Aditya Arora · DSA1 · Action Learning

## 1. Introduction
**Research context.** #8 Lin et al. 2026 (DS-CP), #9 Wang N.-S. et al. 2025 (weights from a domain classifier),
#10 Hu et al. 2026 (conformal guarantees fail under prompt shift), #7 Wang Z. et al. 2025 (detecting shifted
questions); Tibshirani et al. 2019 (weighted conformal).

**Objective.** Measure how far the realised error among answered questions drifts from the target when thresholds
are calibrated on TriviaQA and applied to BioASQ and MMLU medical, and how much of that drift reweighting removes.

**Research question.** With thresholds calibrated on TriviaQA, how large is the gap between realised and target error
under shift, and how much do TF-IDF and DS-CP weights reduce it?

**Hypothesis.** Standard calibration exceeds the target on BioASQ; both weightings narrow the gap; DS-CP narrows it most.

**Shared rule.** All variations use the same rule; only the weights change:
answer if u ≤ t, with t the largest value such that (Σ wᵢ·1[uᵢ≤t]·errᵢ + λ) / (Σ wᵢ·1[uᵢ≤t] + λ) ≤ target.
λ is DS-CP's mass at infinity (λ = 1 for uniform weights, λ = max wᵢ for weighted methods), adapted from
coverage to error among answered questions.

## 2. Setup

In [1]:
# --- Setup (implementation guide, section 2). Run once per session. ------------------------------
import os, sys, subprocess, random, json
from pathlib import Path

REPO_URL = "https://github.com/adiarora28/dsa1-conformal-abstention.git"   # <- set once for the team
ON_KAGGLE = Path("/kaggle").exists()
REPO_DIR = Path("/kaggle/working/dsa1-conformal-abstention") if ON_KAGGLE else Path.cwd().resolve().parent
if ON_KAGGLE and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if ON_KAGGLE:  # upgrade only what Kaggle's image may lack; never reinstall torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.56",
                    "accelerate", "bitsandbytes", "sentence-transformers", "xgboost"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
except ImportError:
    print("PyTorch not installed (this notebook can run on CPU)")

from dsa1 import config as C
from dsa1.cache_io import link_cache_roots
CACHE_ROOT = link_cache_roots("/kaggle/input", "/kaggle/working/cache_merged") if ON_KAGGLE else Path("cache")
print("CACHE_ROOT:", CACHE_ROOT)
if ON_KAGGLE:  # bring outputs of earlier notebooks (attached as inputs) into this working copy
    import shutil
    Path("results").mkdir(exist_ok=True)
    for p in Path("/kaggle/input").rglob("semantic_entropy"):
        if p.is_dir():
            shutil.copytree(p, "results/semantic_entropy", dirs_exist_ok=True)
    for name in ["hand_check.csv", "failure_cases.csv"]:
        hits = sorted(Path("/kaggle/input").rglob(name))
        if hits and not (Path("results") / name).exists():
            shutil.copy(hits[0], Path("results") / name)
            print("copied", hits[0])
for p in sorted(Path(CACHE_ROOT).glob("*/*")):
    print("  ", p.relative_to(CACHE_ROOT))


Cloning into '/kaggle/working/dsa1-conformal-abstention'...


PyTorch 2.11.0+cpu | CUDA: False | CPU
CACHE_ROOT: /kaggle/working/cache_merged
   qwen3-1.7b_fp16/bioasq
   qwen3-1.7b_fp16/mmlu_med
   qwen3-1.7b_fp16/mmlu_other
   qwen3-1.7b_fp16/triviaqa_pool
   qwen3-1.7b_fp16/triviaqa_probe
   qwen3-4b_4bit/bioasq_subset
   qwen3-4b_4bit/triviaqa_pool_subset
   qwen3-4b_8bit/bioasq_subset
   qwen3-4b_8bit/triviaqa_pool_subset
   qwen3-4b_fp16/bioasq
   qwen3-4b_fp16/mmlu_med
   qwen3-4b_fp16/mmlu_other
   qwen3-4b_fp16/triviaqa_pool
   qwen3-4b_fp16/triviaqa_probe
   qwen3-8b_4bit/bioasq
   qwen3-8b_4bit/mmlu_med
   qwen3-8b_4bit/mmlu_other
   qwen3-8b_4bit/triviaqa_pool
   qwen3-8b_4bit/triviaqa_probe


In [2]:
# Score tables written by Task 1 (attach Sai Akhil's notebook output as an input on Kaggle).
# Until they exist, token-probability scores are computed directly from the cache so you can start now.
from dsa1 import experiments as E

def find_scores_root():
    for p in Path("/kaggle/input").rglob("scores") if ON_KAGGLE else []:
        if p.is_dir() and any(p.glob("qwen3-*/*.parquet")):
            return p
    return Path("results/scores")

SCORES_ROOT = find_scores_root()
print("SCORES_ROOT:", SCORES_ROOT)

def get_scores(model, dataset):
    path = Path(SCORES_ROOT) / model / f"{dataset}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    print(f"[{model}/{dataset}] no Task 1 score table yet - using token probability from the cache")
    return E.base_scores(CACHE_ROOT, model, dataset)


SCORES_ROOT: results/scores


In [3]:
from dsa1.splits import load_splits
SPLITS = load_splits("data/splits/splits.json")
SEEDS = [int(s) for s in SPLITS["seeds"]]
TEST_KEYS = {"triviaqa_pool": "test_triviaqa", "bioasq": "test_bioasq", "mmlu_med": "test_mmlu_med"}
print("seeds:", SEEDS, "| sizes (seed 0):", {k: len(v) for k, v in SPLITS["seeds"]["0"].items()})


seeds: [0, 1, 2, 3, 4] | sizes (seed 0): {'cal': 1000, 'test_triviaqa': 1000, 'test_bioasq': 1000, 'test_mmlu_med': 945}


## 3. Data loading & exploration

In [4]:
from dsa1 import data
QTEXT = {}
for t in ["triviaqa", "bioasq", "mmlu_med", "mmlu_other"]:
    try:
        q = data.read_questions(t)
        QTEXT.update(dict(zip(q["qid"], q["question"])))
    except FileNotFoundError:
        pass
if not Path("data/embeddings/triviaqa.npz").exists():
    subprocess.run([sys.executable, "scripts/embed_questions.py"], check=True)
EMB = {}
for t in ["triviaqa", "bioasq", "mmlu_med", "mmlu_other"]:
    if Path(f"data/embeddings/{t}.npz").exists():
        EMB.update(E.load_embeddings(t))
print(len(QTEXT), "questions,", len(EMB), "embeddings")

MODELS, ALL = [], {}
for m in C.MODELS:
    try:
        ALL[m] = pd.concat([get_scores(m, ds) for ds in ["triviaqa_pool", "bioasq", "mmlu_med"]]).set_index("qid")
        MODELS.append(m)
    except FileNotFoundError as e:
        print("skip", m, e)
SCORES = [s for s in ["u_tokprob", "u_se", "u_probe"] if any(ALL[m][s].notna().all() for m in MODELS if s in ALL[m])]
print("models:", MODELS, "| scores:", SCORES)
pd.DataFrame({m: ALL[m].groupby("dataset")["err"].mean() for m in MODELS}).rename_axis("error rate")

9640 questions, 9640 embeddings
[qwen3-1.7b/triviaqa_pool] no Task 1 score table yet - using token probability from the cache
[qwen3-1.7b/bioasq] no Task 1 score table yet - using token probability from the cache
[qwen3-1.7b/mmlu_med] no Task 1 score table yet - using token probability from the cache
[qwen3-4b/triviaqa_pool] no Task 1 score table yet - using token probability from the cache
[qwen3-4b/bioasq] no Task 1 score table yet - using token probability from the cache
[qwen3-4b/mmlu_med] no Task 1 score table yet - using token probability from the cache
[qwen3-8b/triviaqa_pool] no Task 1 score table yet - using token probability from the cache
[qwen3-8b/bioasq] no Task 1 score table yet - using token probability from the cache
[qwen3-8b/mmlu_med] no Task 1 score table yet - using token probability from the cache
models: ['qwen3-1.7b', 'qwen3-4b', 'qwen3-8b'] | scores: ['u_tokprob']


,qwen3-1.7b,qwen3-4b,qwen3-8b
error rate,,,
bioasq,0.958333,0.864897,0.875000
mmlu_med,0.443386,0.291005,0.240212
triviaqa_pool,0.854167,0.652000,0.500000


## 4. Preprocessing – test sets with graded shift
Per seed: mixtures of 1,000 questions with 0/25/50/75/100% BioASQ (rest TriviaQA test), each MMLU medical subject,
and all 945 MMLU medical questions. Weights see only the questions of each test set, never their labels.

In [5]:
from dsa1.splits import mixture
MM = data.read_questions("mmlu_med")
def test_sets(seed):
    d = SPLITS["seeds"][str(seed)]
    sets = {f"mix_{int(f * 100):03d}": mixture(d["test_triviaqa"], d["test_bioasq"], f, n=1000, seed=seed)
            for f in [0, 0.25, 0.5, 0.75, 1.0]}
    sets["mmlu_all"] = d["test_mmlu_med"]
    for subj in C.MMLU_MEDICAL_SUBJECTS:
        sets[f"mmlu_{subj}"] = MM.loc[MM["subject"] == subj, "qid"].tolist()
    return sets
{k: len(v) for k, v in test_sets(0).items()}

{'mix_000': 1000,
 'mix_025': 1000,
 'mix_050': 1000,
 'mix_075': 1000,
 'mix_100': 1000,
 'mmlu_all': 945,
 'mmlu_anatomy': 135,
 'mmlu_clinical_knowledge': 265,
 'mmlu_college_medicine': 173,
 'mmlu_medical_genetics': 100,
 'mmlu_professional_medicine': 272}

## 5. Model definition – the three weightings
- **V1 uniform:** wᵢ = 1, λ = 1 (standard split conformal).
- **V2 TF-IDF:** logistic regression on word 1–2-gram TF-IDF, 5-fold cross-fitted, weights clipped at the 99th percentile, λ = max wᵢ.
- **V3 DS-CP:** XGBoost on all-MiniLM-L6-v2 embeddings with the paper's settings (Appendix C), λ = max wᵢ (Algorithm 1).

In [8]:
from dsa1.weights import uniform_weights, tfidf_weights, dscp_weights, embedding_shift_distance, weighted_conformal_quantile
METHODS = ["uniform", "tfidf", "dscp"]
def make_weights(cal_q, tgt_q, method, **kw):
    if method == "uniform":
        return uniform_weights(len(cal_q))
    if method == "tfidf":
        return tfidf_weights([QTEXT[q] for q in cal_q], [QTEXT[q] for q in tgt_q], **kw)
    if method == "dscp":
        return dscp_weights(E.embed_rows(EMB, cal_q), E.embed_rows(EMB, tgt_q), **kw)
    raise ValueError(method)

## 6. Running the variations

### 6a. Reproduction first: DS-CP coverage on MMLU (90% target)
**Exact reproduction:** clone the authors' code (`https://github.com/zhexiaolin/CP`), run it for one or two models
on the logits they reuse from LLM-Uncertainty-Bench, and confirm standard CP under-covers on some subject pairs while
DS-CP stays closer to 90%. Then run the cell below to check that *our* `weighted_conformal_quantile` + `dscp_weights`
give the same qualitative result with Qwen3 (calibrate on non-medical MMLU, test on each medical subject).

In [ ]:
from dsa1.cache_io import cache_dir, load_records
def lac_scores(rec):  # nonconformity = 1 - probability of the correct option (LAC, as in the paper)
    P = np.stack(rec["option_probs"].values)
    return P, 1 - P[np.arange(len(P)), rec["answer_idx"].to_numpy()]
rows = []
try:
    rec_o = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", "mmlu_other"))
    rec_m = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", "mmlu_med"))
    _, S_cal = lac_scores(rec_o)
    for subj in C.MMLU_MEDICAL_SUBJECTS:
        r = rec_m[rec_m["subject"] == subj]
        P_t, S_t = lac_scores(r)
        W = dscp_weights(E.embed_rows(EMB, rec_o["qid"].tolist()), E.embed_rows(EMB, r["qid"].tolist()))
        for name, w, lam in [("standard CP", np.ones(len(S_cal)), 1.0), ("DS-CP", W.w, W.lam)]:
            q = weighted_conformal_quantile(S_cal, w, lam, alpha=0.10)
            rows.append({"subject": subj, "method": name, "coverage": float((S_t <= q).mean()),
                         "avg_set_size": float(((1 - P_t) <= q).sum(axis=1).mean()), "ess": float(w.sum() ** 2 / (w ** 2).sum())})
    repro = pd.DataFrame(rows)
    repro.to_csv("results/metrics/task2_dscp_reproduction.csv", index=False)
    display(repro.pivot_table(index="subject", columns="method", values=["coverage", "avg_set_size"]).round(3))
except FileNotFoundError as e:
    print("mmlu_other cache not available yet:", e)

### 6b. Main experiment: 3 weightings × 3 targets × all test sets × 5 seeds × scores × models

In [ ]:
import time
rows = []
for s in SEEDS:
    cal_q = SPLITS["seeds"][str(s)]["cal"]
    for name, tgt_q in test_sets(s).items():
        t0 = time.perf_counter()
        Ws = {m: make_weights(cal_q, tgt_q, m) for m in METHODS}
        t_w = time.perf_counter() - t0
        dist = embedding_shift_distance(E.embed_rows(EMB, cal_q), E.embed_rows(EMB, tgt_q))
        for model in MODELS:
            cal, tst = ALL[model].loc[cal_q], ALL[model].loc[tgt_q]
            for sc in SCORES:
                if cal[sc].isna().any() or tst[sc].isna().any():
                    continue
                for m in METHODS:
                    for r in E.shift_eval(cal[sc].values, cal["err"].values, tst[sc].values, tst["err"].values, Ws[m]):
                        rows.append({"seed": s, "test_set": name, "model": model, "score": sc,
                                     "shift_dist": dist, "dscp_auc": Ws["dscp"].domain_auc,
                                     "tfidf_auc": Ws["tfidf"].domain_auc, "weight_time_s": t_w, **r})
    print("seed", s, "done")
res = pd.DataFrame(rows)
res.to_csv("results/metrics/task2_results.csv", index=False)
res.head()

## 7. Evaluation – signed gap (realised − target) and coverage, mean ± std over seeds

In [ ]:
from dsa1.metrics import mean_std
main = res[(res["model"] == "qwen3-4b") & (res["score"] == "u_tokprob")]
display(main.pivot_table(index=["test_set"], columns=["target", "method"], values="gap", aggfunc=mean_std))
display(main.pivot_table(index=["test_set"], columns=["target", "method"], values="coverage", aggfunc=mean_std))
# 95% t-interval over the five seeds: does the gap exclude zero?
from scipy.stats import t as student_t
g = main.groupby(["test_set", "target", "method"])["gap"]
summary = pd.DataFrame({"mean_gap": g.mean(), "half_width": g.std() * student_t.ppf(0.975, g.count() - 1) / np.sqrt(g.count())})
summary["excludes_zero"] = (summary["mean_gap"].abs() > summary["half_width"])
summary.round(3)

## 8. Comparison & analysis

In [ ]:
# Key figure – gap versus shift size (target 10%), Qwen3-4B, token probability
d = main[main["target"] == 0.10]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, x, lab in [(axes[0], "dscp_auc", "domain-classifier AUC (MiniLM + XGBoost)"), (axes[1], "shift_dist", "mean cosine distance")]:
    for m, gm in d.groupby("method"):
        agg = gm.groupby("test_set").agg(x=(x, "mean"), gap=("gap", "mean"), sd=("gap", "std")).sort_values("x")
        ax.errorbar(agg["x"], agg["gap"], yerr=agg["sd"], marker="o", capsize=3, label=m)
    ax.axhline(0, c="grey", ls=":"); ax.set_xlabel(lab); ax.set_ylabel("realised − target error (answered)")
    ax.legend(); ax.set_title("Gap vs shift size, target 10%")
plt.tight_layout(); plt.savefig("results/figures/task2_gap_vs_shift.png", dpi=150); plt.show()

In [ ]:
# Coverage versus gap (what does each method pay for its correction?)
fig, ax = plt.subplots(figsize=(6, 4))
for m, gm in d[d["test_set"].isin(["mix_100", "mmlu_all"])].groupby("method"):
    ax.scatter(gm["coverage"], gm["gap"], label=m, alpha=0.7)
ax.axhline(0, c="grey", ls=":"); ax.set_xlabel("coverage (share answered)"); ax.set_ylabel("gap"); ax.legend()
plt.tight_layout(); plt.savefig("results/figures/task2_coverage_vs_gap.png", dpi=150); plt.show()

In [ ]:
# Weight histograms on 100% BioASQ, seed 0
cal_q = SPLITS["seeds"]["0"]["cal"]; tgt_q = test_sets(0)["mix_100"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, m in zip(axes, ["tfidf", "dscp"]):
    W = make_weights(cal_q, tgt_q, m)
    ax.hist(np.log10(W.w + 1e-12), bins=50); ax.set_title(f"{m}: ESS = {W.ess:.0f} of {len(W.w)}, AUC = {W.domain_auc:.3f}")
    ax.set_xlabel("log10 weight")
plt.tight_layout(); plt.savefig("results/figures/task2_weight_hist.png", dpi=150); plt.show()

In [ ]:
# Ablations on 100% BioASQ: TF-IDF clipping and DS-CP lambda (gamma × max weight), token probability, Qwen3-4B
rows = []
for s in SEEDS:
    cal_q = SPLITS["seeds"][str(s)]["cal"]; tgt_q = test_sets(s)["mix_100"]
    cal, tst = ALL["qwen3-4b"].loc[cal_q], ALL["qwen3-4b"].loc[tgt_q]
    variants = {f"tfidf clip={c}": make_weights(cal_q, tgt_q, "tfidf", clip_quantile=c) for c in [None, 0.99, 0.95]}
    variants.update({f"dscp gamma={g}": make_weights(cal_q, tgt_q, "dscp", gamma=g) for g in [0.5, 1.0, 2.0]})
    for name, W in variants.items():
        for r in E.shift_eval(cal["u_tokprob"].values, cal["err"].values, tst["u_tokprob"].values, tst["err"].values, W):
            rows.append({"seed": s, "variant": name, **r})
abl = pd.DataFrame(rows)
abl.to_csv("results/metrics/task2_ablations.csv", index=False)
abl.pivot_table(index="variant", columns="target", values=["gap", "coverage", "ess"], aggfunc="mean").round(3)

In [ ]:
# Shift alarm (SConU-style): share of test questions with conformal p-value < 0.05, vs share of BioASQ
from dsa1.weights import conformal_pvalues
rows = []
for s in SEEDS:
    cal_q = SPLITS["seeds"][str(s)]["cal"]
    for name, tgt_q in test_sets(s).items():
        if not name.startswith("mix_"):
            continue
        p = conformal_pvalues(ALL["qwen3-4b"].loc[cal_q, "u_tokprob"].values, ALL["qwen3-4b"].loc[tgt_q, "u_tokprob"].values)
        rows.append({"seed": s, "share_bioasq": int(name[-3:]) / 100, "flagged": float((p < 0.05).mean())})
alarm = pd.DataFrame(rows)
agg = alarm.groupby("share_bioasq")["flagged"].agg(["mean", "std"])
plt.errorbar(agg.index, agg["mean"], yerr=agg["std"], marker="o", capsize=3); plt.axhline(0.05, ls=":", c="grey")
plt.xlabel("share of BioASQ questions"); plt.ylabel("share flagged (p < 0.05)"); plt.title("Shift alarm")
plt.savefig("results/figures/task2_shift_alarm.png", dpi=150); plt.show()

### Confusion matrix and demo (for the defense)

In [ ]:
# Confusion matrix: answered/abstained × right/wrong on 100% BioASQ, seed 0, target 10%, each weighting
from dsa1.rules import weighted_plugin_threshold
cal_q = SPLITS["seeds"]["0"]["cal"]; tgt_q = test_sets(0)["mix_100"]
cal, tst = ALL["qwen3-4b"].loc[cal_q], ALL["qwen3-4b"].loc[tgt_q]
panels, rows, thresholds = [], [], {}
for m in METHODS:
    W = make_weights(cal_q, tgt_q, m)
    t = weighted_plugin_threshold(cal["u_tokprob"].values, cal["err"].values, 0.10, w=W.w, lam=W.lam)
    thresholds[m] = t
    c = E.abstention_confusion(tst["u_tokprob"], tst["err"], t)
    panels.append((m, c)); rows.append({"method": m, "threshold": t, **c})
pd.DataFrame(rows).to_csv("results/metrics/task2_confusion.csv", index=False)
E.plot_confusions(panels, "abstention", "results/figures/task2_confusion.png", "Qwen3-4B · BioASQ · target 10% · seed 0")
plt.show()
pd.DataFrame(rows)

In [ ]:
# Demo – inference on test samples: five BioASQ questions, the model's answer, and each method's decision
from dsa1.cache_io import cache_dir, load_records
pd.set_option("display.max_colwidth", 70)
rec = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", "bioasq")).set_index("qid")
pick = tst.sample(5, random_state=3)
demo = pd.DataFrame({
    "question": [rec.loc[q, "question"] for q in pick.index],
    "model answer": [rec.loc[q, "answer"] for q in pick.index],
    "correct": (pick["err"] == 0).values,
    "uncertainty": pick["u_tokprob"].round(3).values,
    **{m: np.where(pick["u_tokprob"] <= thresholds[m], "answer", "route to staff") for m in METHODS},
})
demo.to_csv("results/metrics/task2_demo_examples.csv", index=False)
demo

## 9. Discussion
- How large is the gap without correction on BioASQ and MMLU medical, at each target?
- Does weighting reduce it? Does DS-CP beat the TF-IDF classifier? At what coverage cost?
- What do effective sample size and the weight histograms say about overlap between TriviaQA and BioASQ?
- Did the reproduction match the DS-CP paper qualitatively?
- Limitations: λ-adaptation to error among answered is ours; exact-match labels; one model family.
- MVP: which correction to use and whether the shift alarm can drive monitoring and recalibration.

## 10. References

- Bibliography numbers refer to `DSA 1 - Bibliography.xlsx`.
- Datasets: TriviaQA (Joshi et al., 2017), BioASQ Task B 14b (Krithara et al., 2023), MMLU (Hendrycks et al., 2021).
- Models: Qwen3 (Qwen Team, 2025); DeBERTa-large-MNLI (He et al., 2021); all-MiniLM-L6-v2 (Reimers & Gurevych, 2019).
- Code: shared `dsa1` package in this repository; LM-Polygraph (Vashurin et al., 2025) used for cross-checks;
  DS-CP reference code https://github.com/zhexiaolin/CP (Lin et al., 2026).
